# Uniform Cost Search (UCS) and A* Search

This notebook demonstrates two informed/uninformed graph-search algorithms using a weighted graph.

- **UCS (Uniform Cost Search):** expands the frontier node with the lowest accumulated path cost `g(n)`.
- **A* Search:** expands the frontier node with the lowest `f(n) = g(n) + h(n)`, where `h(n)` estimates the remaining cost.

Note: “UCL” is not the usual name of this search algorithm; this notebook assumes you meant **UCS**.

In [ ]:
import heapq
from itertools import count

# Weighted, undirected graph: node -> [(neighbor, edge_cost), ...]
graph = {
    'S': [('A', 1), ('B', 4)],
    'A': [('S', 1), ('B', 2), ('C', 5)],
    'B': [('S', 4), ('A', 2), ('C', 1), ('G', 7)],
    'C': [('A', 5), ('B', 1), ('G', 3)],
    'G': [('B', 7), ('C', 3)]
}

# Admissible heuristic estimates to goal G (never overestimate the true cost)
heuristic = {'S': 7, 'A': 6, 'B': 4, 'C': 3, 'G': 0}


## 1. Uniform Cost Search (UCS)

UCS is equivalent to Dijkstra's algorithm for non-negative edge costs. It does not use a heuristic.

In [ ]:
def uniform_cost_search(graph, start, goal):
    tie_breaker = count()
    frontier = [(0, next(tie_breaker), start, [start])]
    best_cost = {start: 0}
    expanded = []

    while frontier:
        cost, _, node, path = heapq.heappop(frontier)

        # Ignore outdated queue entries
        if cost != best_cost.get(node, float('inf')):
            continue

        expanded.append(node)
        if node == goal:
            return path, cost, expanded

        for neighbor, edge_cost in graph.get(node, []):
            if edge_cost < 0:
                raise ValueError('UCS requires non-negative edge costs.')
            new_cost = cost + edge_cost
            if new_cost < best_cost.get(neighbor, float('inf')):
                best_cost[neighbor] = new_cost
                heapq.heappush(
                    frontier,
                    (new_cost, next(tie_breaker), neighbor, path + [neighbor])
                )

    return None, float('inf'), expanded

ucs_path, ucs_cost, ucs_expanded = uniform_cost_search(graph, 'S', 'G')
print('UCS path:', ucs_path)
print('UCS total cost:', ucs_cost)
print('UCS expanded nodes:', ucs_expanded)


## 2. A* Search

A* prioritizes nodes using `f(n) = g(n) + h(n)`. With an admissible heuristic (and the graph-search implementation below reopening improved paths), A* can find an optimal path.

In [ ]:
def a_star_search(graph, start, goal, heuristic):
    tie_breaker = count()
    start_h = heuristic.get(start, 0)
    frontier = [(start_h, next(tie_breaker), 0, start, [start])]
    best_g = {start: 0}
    expanded = []

    while frontier:
        f_score, _, g_cost, node, path = heapq.heappop(frontier)

        # Skip entries that no longer represent the best known path
        if g_cost != best_g.get(node, float('inf')):
            continue

        expanded.append(node)
        if node == goal:
            return path, g_cost, expanded

        for neighbor, edge_cost in graph.get(node, []):
            if edge_cost < 0:
                raise ValueError('A* requires non-negative edge costs.')
            tentative_g = g_cost + edge_cost
            if tentative_g < best_g.get(neighbor, float('inf')):
                best_g[neighbor] = tentative_g
                h_cost = heuristic.get(neighbor, 0)
                heapq.heappush(
                    frontier,
                    (tentative_g + h_cost, next(tie_breaker), tentative_g,
                     neighbor, path + [neighbor])
                )

    return None, float('inf'), expanded

astar_path, astar_cost, astar_expanded = a_star_search(
    graph, 'S', 'G', heuristic
)
print('A* path:', astar_path)
print('A* total cost:', astar_cost)
print('A* expanded nodes:', astar_expanded)


## 3. Compare the results

Both algorithms should return an optimal path with the same total cost for this graph. A* may expand a different number/order of nodes depending on the heuristic.

In [ ]:
assert ucs_path is not None and astar_path is not None
assert ucs_cost == astar_cost
print(f'UCS cost = {ucs_cost}; A* cost = {astar_cost}')
print('Optimal costs match:', ucs_cost == astar_cost)


## Key differences

| Property | UCS | A* |
|---|---|---|
| Priority | `g(n)` | `g(n) + h(n)` |
| Heuristic | Not used | Used |
| Optimality | Yes, for non-negative edge costs | Yes under standard conditions, including an admissible heuristic for this graph-search approach |
| Typical behavior | May explore many low-cost paths | A useful heuristic can guide search toward the goal |

Run the cells from top to bottom in Jupyter Notebook or Google Colab.